In [1]:
!pip install xgboost



   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   -- ------------------------------------- 0.8/12.6 MB 6.7 MB/s eta 0:00:02
   ------ --------------------------------- 2.1/12.6 MB 4.5 MB/s eta 0:00:03
   --------- ------------------------------ 2.9/12.6 MB 4.3 MB/s eta 0:00:03
   ----------- ---------------------------- 3.7/12.6 MB 4.2 MB/s eta 0:00:03
   -------------- ------------------------- 4.5/12.6 MB 4.1 MB/s eta 0:00:02
   ---------------- ----------------------- 5.2/12.6 MB 4.1 MB/s eta 0:00:02
   ------------------- -------------------- 6.3/12.6 MB 4.1 MB/s eta 0:00:02
   ---------------------- ----------------- 7.1/12.6 MB 4.0 MB/s eta 0:00:02
   ------------------------ --------------- 7.6/12.6 MB 4.0 MB/s eta 0:00:02
   --------------------------- ------------ 8.7/12.6 MB 4.0 MB/s eta 0:00:01
   ------------------------------ --------- 9.7/12.6 MB 4.0 MB/s eta 0:00:01
   --------------------------------- ------ 10.5/12.6 MB 4.0 MB/s eta 0:00:01
   --

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.2.6 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.2.6 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.6 which is incompatible.
streamlit 1.37.1 requires pillow<11,>=7.1.0, but you have pillow 11.3.0 which is incompatible.
tensorflow-intel 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.


In [2]:
!pip install tensorflow


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

import re

from sklearn.preprocessing import StandardScaler, MinMaxScaler

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2, f_classif, mutual_info_classif,f_regression

from sklearn.model_selection import train_test_split

import xgboost as xgb

import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

from tensorflow.keras.layers import Dense, Dropout, Activation
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import SGDRegressor
from xgboost import XGBRegressor

from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.models import Sequential
from math import sqrt
from sklearn.metrics import mean_squared_error, r2_score, median_absolute_error
from sklearn.metrics import mean_absolute_error

import warnings
warnings.filterwarnings('ignore')


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
#Uploading dataset

In [ ]:
df=pd.read_csv("SolarPrediction.csv")


In [ ]:
#datapreprocessing

In [ ]:
df.head(5)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
print(df.dtypes)

In [ ]:
df.describe()

In [ ]:
# Convert 'Data', 'Time', 'TimeSunRise', 'TimeSunSet' to datetime format
df['Data'] = pd.to_datetime(df['Data'], errors='coerce')
df['Time'] = pd.to_datetime(df['Time'], errors='coerce')
df['TimeSunRise'] = pd.to_datetime(df['TimeSunRise'], errors='coerce')
df['TimeSunSet'] = pd.to_datetime(df['TimeSunSet'], errors='coerce')


In [ ]:
# Extract useful features from 'TimeSunRise' and 'TimeSunSet'
df['Sunrise_Hour'] = df['TimeSunRise'].dt.hour
df['Sunrise_Minute'] = df['TimeSunRise'].dt.minute
df['Sunset_Hour'] = df['TimeSunSet'].dt.hour
df['Sunset_Minute'] = df['TimeSunSet'].dt.minute
df['Hour'] = df['Time'].dt.hour


In [ ]:
df.drop(['UNIXTime','Data','Time','TimeSunRise','TimeSunSet'],axis=1,inplace=True)

In [ ]:
# Features and target
features = ['Temperature', 'Pressure', 'Humidity', 'WindDirection(Degrees)', 'Speed']
X = df[features]
y = df['Radiation']

In [ ]:
cor=df.corr()
plt.figure(figsize=(7,4))
sns.heatmap(cor,annot=True)
plt.show()

In [ ]:
#Feature Selection

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
df.dropna(inplace=True)
X = df.drop(columns=['Radiation'])
y = df['Radiation']

In [ ]:
# 3. Feature Engineering — creating interaction features before selection
X['Temp_Speed'] = X['Temperature'] * X['Speed']
X['Humidity_Pressure'] = X['Humidity'] * X['Pressure']

In [ ]:
# 4. Feature Selection
selector = SelectKBest(score_func=f_regression, k=6)
X_selected = selector.fit_transform(X, y)
selected_columns = X.columns[selector.get_support()]


In [ ]:
X = X[selected_columns]

# 5. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# 7. Random Forest
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train_scaled, y_train)
rf_pred = rf.predict(X_test_scaled)

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(y_test, rf_pred, color='royalblue', alpha=0.6)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         'r--')

plt.xlabel("Actual Radiation")
plt.ylabel("Predicted Radiation")
plt.title("Random Forest: Actual vs Predicted")
plt.tight_layout()

plt.savefig("random_forest_actual_vs_predicted.png", dpi=300)
plt.show()

In [ ]:
# 8. XGBoost
xgb = XGBRegressor(random_state=42)
xgb.fit(X_train_scaled, y_train)
xgb_pred = xgb.predict(X_test_scaled)

In [ ]:
# 9. SGD Regressor
sgd = SGDRegressor(random_state=42, max_iter=1000, tol=1e-3)
sgd.fit(X_train_scaled, y_train)
sgd_pred = sgd.predict(X_test_scaled)

In [ ]:
# 10. Adam-based Neural Network
model = Sequential([
    Dense(64, activation='relu', input_dim=X_train_scaled.shape[1]),
    Dense(64, activation='relu'),
    Dense(1)
])
model.compile(optimizer='adam', loss='mse')
model.fit(X_train_scaled, y_train, epochs=100, verbose=0)
adam_pred = model.predict(X_test_scaled).flatten()


In [ ]:
# 11. Evaluation
def print_metrics(name, y_test, y_pred):
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    print(f"{name} RMSE: {rmse:.2f}")
    print(f"{name} R²: {r2:.2f}")
    print(f"{name} MSE: {mse:.2f}\n")

In [ ]:
print_metrics("Random Forest", y_test, rf_pred)
print_metrics("XGBoost", y_test, xgb_pred)
print_metrics("SGD", y_test, sgd_pred)
print_metrics("Adam", y_test, adam_pred)

In [ ]:
# 12. Visualization: Actual vs Predicted
plt.figure(figsize=(10, 6))
plt.scatter(y_test, rf_pred, label='Random Forest', alpha=0.6)
plt.scatter(y_test, xgb_pred, label='XGBoost', alpha=0.6)
plt.scatter(y_test, sgd_pred, label='SGD', alpha=0.6)
plt.scatter(y_test, adam_pred, label='Adam', alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
plt.xlabel('Actual Radiation')
plt.ylabel('Predicted Radiation')
plt.title('Actual vs Predicted Radiation')
plt.legend()
plt.show()

In [ ]:
# 13. Distribution Plot
plt.figure(figsize=(10, 6))
sns.histplot(y_test - rf_pred, kde=True, label='Random Forest', color='blue', alpha=0.5)
sns.histplot(y_test - xgb_pred, kde=True, label='XGBoost', color='green', alpha=0.5)
sns.histplot(y_test - sgd_pred, kde=True, label='SGD', color='orange', alpha=0.5)
sns.histplot(y_test - adam_pred, kde=True, label='Adam', color='purple', alpha=0.5)
plt.legend()
plt.title('Error Distribution')
plt.show()

In [ ]:
plt.figure(figsize=(16, 12))

# Random Forest
plt.subplot(2, 2, 1)
plt.scatter(y_test, rf_pred, color='royalblue', alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Random Forest: Actual vs Predicted')

# XGBoost
plt.subplot(2, 2, 2)
plt.scatter(y_test, xgb_pred, color='darkorange', alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('XGBoost: Actual vs Predicted')

# SGD
plt.subplot(2, 2, 3)
plt.scatter(y_test, sgd_pred, color='seagreen', alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('SGD: Actual vs Predicted')

# Adam
plt.subplot(2, 2, 4)
plt.scatter(y_test, adam_pred.flatten(), color='mediumpurple', alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Adam: Actual vs Predicted')

plt.tight_layout()
plt.show()

In [ ]:
sample_indices = np.random.choice(len(y_test), size=20, replace=False)
sample_actual = y_test.iloc[sample_indices]
sample_pred = xgb_pred[sample_indices]

plt.subplot(2, 2, 2)
bar_width = 0.35
index = np.arange(len(sample_indices))

plt.bar(index, sample_actual, bar_width, label='Actual', color='steelblue')
plt.bar(index + bar_width, sample_pred, bar_width, label='Predicted', color='darkorange')
plt.xlabel('Sample Index')
plt.ylabel('Radiation')
plt.title('XGBoost: Actual vs Predicted (Bar Plot)')
plt.legend()




In [ ]:
models = ['Random Forest', 'XGBoost', 'Adam (Neural Network)', 'SGD (SGD)']
accuracy = [75, 73, 71, 56]

# Bar colors (using custom colors for each model)
colors = [
          '#FFD700',  # bold gold
          '#8A2BE2',  # strong purple
          '#00CED1',  # turquoise
          
          '#FF1493',]  # deep pink
         

# Create the bar plot
plt.figure(figsize=(8, 5))  # Adjust figure size
bars = plt.bar(models, accuracy, color=colors)

# Adding labels and title
plt.xlabel('Model')
plt.ylabel('Accuracy (%)')
plt.title('Model Accuracy Comparison')

# Display the value of each bar on top of the bar
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, yval + 0, f'{yval}%', ha='center', va='bottom', fontsize=12)

# Show the plot
plt.show()
